<a href="https://colab.research.google.com/github/myNameDyK/GEMM_32x32_KV260_AI_ACCELERATOR_CORE_FPGA/blob/SPARSITY_ACCELERATOR/OFFICIAL_SPARSITY_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece datasets==2.21.0 rouge_score

In [ ]:
import torch
import math
import types
import gc
import re
import string
import pandas as pd

from datasets import load_dataset
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

ALPHA = 20
SEQ_LEN = 2048

print(torch.cuda.get_device_name(0))


Tesla T4


In [ ]:
# Clone LongBench repository

import os

if not os.path.exists("/content/LongBench"):
    !git clone https://github.com/THUDM/LongBench.git /content/LongBench
else:
    print("LongBench already exists")

LongBench already exists


In [ ]:
from huggingface_hub import login

login()

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"
)

model.eval()

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

In [ ]:
from datasets import load_dataset


tasks = [
    "narrativeqa",
    "qasper",
    "gov_report"
]

SAMPLES_PER_TASK = 15
MAX_TOTAL_SAMPLES = 50

samples = []

for task in tasks:

    print("="*50)
    print("Loading:", task)

    ds = load_dataset(
        "THUDM/LongBench",
        task,
        split="test"
    )

    print(ds)
    print("Fields:", ds.column_names)

    for item in ds.select(range(min(SAMPLES_PER_TASK, len(ds)))):
        samples.append({
            "task": task,
            "context": item["context"],
            "input": item["input"],
            "answers": item["answers"],
            "all_classes": item.get("all_classes", []),
            "length": item.get("length", None)
        })

samples = samples[:MAX_TOTAL_SAMPLES]

print("="*50)
print("TOTAL SAMPLES:", len(samples))
from collections import Counter
print("BY TASK:", Counter(s["task"] for s in samples))
print(samples[0])


Loading: narrativeqa


The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id'],
    num_rows: 200
})
Fields: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id']
Loading: qasper


Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id'],
    num_rows: 200
})
Fields: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id']
Loading: gov_report


Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id'],
    num_rows: 200
})
Fields: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id']
TOTAL SAMPLES: 45
BY TASK: Counter({'narrativeqa': 15, 'qasper': 15, 'gov_report': 15})
{'task': 'narrativeqa', 'context': 'Transcribed from the 1915 Martin Secker edition by David Price, email\nccx074@pglaf.org\n\n                          [Picture: Book cover]\n\n\n\n\n\n                                   THE\n                                COXON FUND\n\n\n                              BY HENRY JAMES\n\n                      [Picture: Decorative graphic]\n\n                                * * * * *\n\n                          LONDON: MARTIN SECKER\n                     NUMBER FIVE JOHN STREET ADELPHI\n\n                                * * * * *\n\n                    This edition first published 1915\n\n                       The text follows that of the\n  

In [ ]:
from transformers.models.llama.modeling_llama import LlamaAttention

# restore toàn bộ attention forward về class gốc
for layer in model.model.layers:

    layer.self_attn.forward = (
        LlamaAttention.forward.__get__(
            layer.self_attn,
            type(layer.self_attn)
        )
    )

print(model.model.layers[0].self_attn.forward)

<bound method LlamaAttention.forward of LlamaAttention(
  (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
)>


In [ ]:
# ============================================================
# PAPER-BASED BLOCK PRUNING
# ============================================================

def get_pruning_mask(attn_weights, block_rows, block_cols, alpha=ALPHA):
    """Return KEEP mask using the paper's row-sum block rule.

    A block is pruned iff every row inside that block has a
    sum smaller than t = alpha / n, where n is the attention
    key/context length.
    """
    rows, cols = attn_weights.shape
    n = cols

    num_block_rows = rows // block_rows
    num_block_cols = cols // block_cols

    if num_block_rows == 0 or num_block_cols == 0:
        return torch.zeros(
            num_block_rows, num_block_cols,
            device=attn_weights.device, dtype=torch.bool
        )

    A = attn_weights[
        :num_block_rows * block_rows,
        :num_block_cols * block_cols
    ]

    blocks = A.reshape(
        num_block_rows, block_rows,
        num_block_cols, block_cols
    )

    # Sum over the block columns, leaving one value per row.
    row_sums = blocks.sum(dim=-1)

    threshold = alpha / n

    # True = prune.  We return the inverse because the hook
    # uses True = keep.
    prune_mask = (row_sums < threshold).all(dim=1)
    keep_mask = ~prune_mask

    return keep_mask


In [ ]:
# ============================================================
# Sparse Attention Hook - PAPER-COMPATIBLE + CACHE SAFE
# ============================================================

# The previous hook rebuilt attention but returned (out, None).
# That loses the KV cache during generation and can both slow down
# inference and change the generated text substantially.
#
# This version:
# 1. preserves RoPE;
# 2. preserves attention_mask;
# 3. supports both legacy tuple KV-cache and newer Cache API;
# 4. performs paper block pruning on the prefill attention matrix;
# 5. leaves decode attention dense while still preserving the cache;
# 6. returns the same output structure expected by the installed
#    Transformers version.

sparse_stats = {
    "sum_sparsity": 0.0,
    "count": 0
}


def remove_sparse_attention(model):
    for layer in model.model.layers:
        attn = layer.self_attn
        if hasattr(attn, "_original_forward"):
            attn.forward = attn._original_forward

    print("Sparse attention removed")


def _get_past_from_kwargs(kwargs):
    # Transformers 4.x uses past_key_value; newer versions use
    # past_key_values.
    if "past_key_values" in kwargs:
        return kwargs.get("past_key_values"), "past_key_values"
    return kwargs.get("past_key_value"), "past_key_value"


def install_sparse_attention(model, block_rows, block_cols):

    remove_sparse_attention(model)

    sparse_stats["sum_sparsity"] = 0.0
    sparse_stats["count"] = 0

    for layer_id, layer in enumerate(model.model.layers):

        attn = layer.self_attn

        if not hasattr(attn, "_original_forward"):
            attn._original_forward = attn.forward

        original_forward = attn._original_forward

        def make_forward(original, layer_id):

            def forward(self, hidden_states, *args, **kwargs):

                bsz, q_len, _ = hidden_states.shape

                # ----------------------------------------------------
                # Read arguments in a version-tolerant way.
                # ----------------------------------------------------
                attention_mask = kwargs.get("attention_mask", None)
                position_ids = kwargs.get("position_ids", None)
                position_embeddings = kwargs.get("position_embeddings", None)
                cache_position = kwargs.get("cache_position", None)
                use_cache = kwargs.get("use_cache", False)
                past, past_name = _get_past_from_kwargs(kwargs)

                # ----------------------------------------------------
                # QKV
                # ----------------------------------------------------
                q = self.q_proj(hidden_states)
                k = self.k_proj(hidden_states)
                v = self.v_proj(hidden_states)

                nh = getattr(self, "num_heads", self.config.num_attention_heads)
                nk = getattr(
                    self, "num_key_value_heads",
                    self.config.num_key_value_heads
                )
                hd = getattr(
                    self, "head_dim",
                    self.config.hidden_size // nh
                )

                q = q.view(bsz, q_len, nh, hd).transpose(1, 2)
                k = k.view(bsz, q_len, nk, hd).transpose(1, 2)
                v = v.view(bsz, q_len, nk, hd).transpose(1, 2)

                # ----------------------------------------------------
                # RoPE
                # ----------------------------------------------------
                if position_embeddings is not None:
                    cos, sin = position_embeddings
                elif hasattr(self, "rotary_emb") and position_ids is not None:
                    # Compatibility path for Transformers versions where
                    # RoPE is still computed inside LlamaAttention.
                    try:
                        cos, sin = self.rotary_emb(v, position_ids)
                    except TypeError:
                        cos, sin = self.rotary_emb(v, seq_len=k.shape[-2])
                else:
                    cos = sin = None

                if cos is not None and sin is not None:
                    q, k = apply_rotary_pos_emb(q, k, cos, sin)

                # ----------------------------------------------------
                # KV cache: support both Cache objects and old tuples.
                # ----------------------------------------------------
                legacy_cache = False
                past_len = 0

                if past is not None:
                    if hasattr(past, "get_seq_length"):
                        try:
                            past_len = past.get_seq_length(layer_id)
                        except TypeError:
                            past_len = past.get_seq_length()

                        cache_kwargs = {}
                        if cos is not None:
                            cache_kwargs["cos"] = cos
                        if sin is not None:
                            cache_kwargs["sin"] = sin
                        if cache_position is not None:
                            cache_kwargs["cache_position"] = cache_position

                        if cache_kwargs:
                            k, v = past.update(
                                k, v, layer_id, cache_kwargs
                            )
                        else:
                            k, v = past.update(
                                k, v, layer_id
                            )
                    else:
                        # Legacy tuple: (key, value).
                        legacy_cache = True
                        past_k, past_v = past[0], past[1]
                        past_len = past_k.shape[-2]
                        k = torch.cat([past_k, k], dim=2)
                        v = torch.cat([past_v, v], dim=2)

                # ----------------------------------------------------
                # GQA / MQA
                # ----------------------------------------------------
                repeat = nh // nk
                if repeat > 1:
                    k = k.repeat_interleave(repeat, dim=1)
                    v = v.repeat_interleave(repeat, dim=1)

                key_len = k.shape[-2]

                # ----------------------------------------------------
                # QK^T / sqrt(d)
                # ----------------------------------------------------
                scores = torch.matmul(
                    q, k.transpose(-2, -1)
                ) / math.sqrt(hd)

                # Preserve the mask generated by LlamaModel when available.
                if attention_mask is not None:
                    if attention_mask.dim() == 4:
                        scores = scores + attention_mask[:, :, :, :key_len]
                    elif attention_mask.dim() == 2:
                        scores = scores.masked_fill(
                            attention_mask[:, None, None, :key_len] == 0,
                            float("-inf")
                        )
                else:
                    # Fallback causal mask for older Transformers.
                    diagonal = max(key_len - q_len, 0) + 1
                    causal = torch.triu(
                        torch.ones(
                            q_len, key_len,
                            device=scores.device,
                            dtype=torch.bool
                        ),
                        diagonal=diagonal
                    )
                    scores = scores.masked_fill(
                        causal, float("-inf")
                    )

                A = torch.softmax(
                    scores.float(), dim=-1
                ).to(q.dtype)

                # ----------------------------------------------------
                # Paper pruning is meaningful for the square prefill
                # matrix.  Decode (q_len=1) stays dense.
                # ----------------------------------------------------
                do_prune = (
                    q_len > 1
                    and past_len == 0
                    and key_len == q_len
                    and q_len >= block_rows
                    and key_len >= block_cols
                )

                if do_prune:
                    As = A.clone()

                    for h in range(A.shape[1]):
                        head_A = A[0, h]

                        keep_mask = get_pruning_mask(
                            head_A,
                            block_rows,
                            block_cols,
                            ALPHA
                        )

                        if keep_mask.numel() == 0:
                            continue

                        sparsity = (
                            1.0
                            - keep_mask.float().mean().item()
                        )

                        sparse_stats["sum_sparsity"] += sparsity
                        sparse_stats["count"] += 1

                        rows = keep_mask.shape[0] * block_rows
                        cols = keep_mask.shape[1] * block_cols

                        block_view = head_A[:rows, :cols].reshape(
                            keep_mask.shape[0],
                            block_rows,
                            keep_mask.shape[1],
                            block_cols
                        )

                        block_view = block_view * (
                            keep_mask[:, None, :, None]
                        )

                        As[0, h, :rows, :cols] = block_view.reshape(
                            rows, cols
                        )
                else:
                    As = A

                # ----------------------------------------------------
                # AV
                # ----------------------------------------------------
                out = torch.matmul(As.to(v.dtype), v)
                out = out.transpose(1, 2).contiguous()
                out = out.reshape(bsz, q_len, nh * hd)
                out = self.o_proj(out.to(hidden_states.dtype))

                # ----------------------------------------------------
                # Return format. Newer Transformers mutate Cache in place
                # and expect (output, attention_weights). Legacy Llama2
                # expects (output, attention_weights, present_key_value).
                # ----------------------------------------------------
                if legacy_cache:
                    present = (k, v) if use_cache else None
                    return out, None, present

                return out, None

            return forward

        attn.forward = types.MethodType(
            make_forward(original_forward, layer_id),
            attn
        )

    print(
        f"Sparse attention installed: {block_rows}x{block_cols}"
    )


In [ ]:
def generate_longbench_answer(sample):

    task = sample["task"]

    prompt = dataset2prompt[task].format(**sample)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=SEQ_LEN
    ).to(model.device)

    max_new_tokens = 64
    if "dataset2maxlen" in globals():
        max_new_tokens = int(dataset2maxlen.get(task, 64))

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


In [ ]:
# ============================================================
# Export LongBench Prediction - Official Format
# ============================================================

from collections import defaultdict


def export_longbench_prediction(
    mode,
    block=None
):

    # -------------------------
    # Install / remove sparse
    # -------------------------

    if mode == "dense":

        remove_sparse_attention(model)

    else:

        install_sparse_attention(
            model,
            block[0],
            block[1]
        )


    records = []


    # -------------------------
    # Generate predictions
    # -------------------------

    for idx, sample in enumerate(samples):

        print(
            mode,
            idx
        )


        pred = generate_longbench_answer(sample)


        records.append(
            {
                "pred": pred,
                "answers": sample["answers"],
                "task": sample["task"],
                "all_classes": sample.get(
                    "all_classes",
                    []
                ),
                "length": sample.get(
                    "length",
                    None
                )
            }
        )


    # -------------------------
    # Save by dataset/task
    # LongBench official format:
    #
    # pred/
    #   model_name/
    #       qasper.jsonl
    #       gov_report.jsonl
    #       narrativeqa.jsonl
    # -------------------------


    if mode == "dense":
        save_dir = "pred/dense"
    else:
        save_dir = f"pred/sparse_{block[0]}x{block[1]}"

    os.makedirs(
        save_dir,
        exist_ok=True
    )


    grouped_records = defaultdict(list)


    for item in records:

        grouped_records[
            item["task"]
        ].append(item)



    for task, task_records in grouped_records.items():

        filename = f"{save_dir}/{task}.jsonl"


        with open(
            filename,
            "w",
            encoding="utf-8"
        ) as f:

            for item in task_records:

                f.write(
                    json.dumps(
                        item,
                        ensure_ascii=False
                    )
                    + "\n"
                )


        print(
            "Saved:",
            filename
        )

In [ ]:
# Final output:
#
# Dense vs Sparse:
# - sparsity
# - score
# - score loss
#
# This notebook is intended for internal 10-sample LongBench comparison.


In [ ]:
def save_prediction_jsonl(records, filename):

    with open(filename, "w") as f:
        for item in records:
            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                ) + "\n"
            )




In [ ]:
!pip install -q fuzzywuzzy python-Levenshtein rouge

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.8 MB/s eta 0:00:00


# Task A/B/C Block Size Sweep Framework

The original fixed block list is replaced by three exploration tasks.

Every task automatically includes:
- 3x20 : paper baseline
- 32x32 : hardware reference

Task selection is interactive before running.

A : Row expansion (Nx20)
B : Column expansion (3xN)
C : 2D expansion (NxM)


In [ ]:
# Block-size search space

BASELINE_BLOCKS = [
    (3,20),
    (32,32)
]


TASK_A_ROW = [
    (1,20),
    (2,20),
    (3,20),
    (5,20),
    (8,20),
    (10,20),
    (16,20),
    (20,20),
    (32,20),
    (40,20),
    (50,20),
    (64,20),
    (80,20),
    (100,20),
    (32,32)
]


TASK_B_COL = [
    (3,3),
    (3,5),
    (3,10),
    (3,15),
    (3,20),
    (3,30),
    (3,40),
    (3,50),
    (3,64),
    (3,80),
    (3,100),
    (3,128),
    (3,160),
    (3,200),
    (32,32)
]


TASK_C_2D = [
    (1,10),
    (2,20),
    (3,20),
    (4,32),
    (8,32),
    (16,32),
    (16,16),
    (32,32),
    (32,64),
    (32,100),
    (64,64),
    (64,100),
    (100,100),
    (128,128)
]


def prepare_block_list(task):

    if task == "A":
        blocks = TASK_A_ROW

    elif task == "B":
        blocks = TASK_B_COL

    elif task == "C":
        blocks = TASK_C_2D

    else:
        raise ValueError("Choose A, B, or C")


    for b in BASELINE_BLOCKS:
        if b not in blocks:
            blocks.append(b)


    return list(dict.fromkeys(blocks))


In [ ]:
# Select experiment before running

TASK = input(
    "Select task (A=row, B=column, C=2D): "
).upper()


BLOCK_SWEEP = prepare_block_list(TASK)


print("Selected task:", TASK)
print("Number of block sizes:", len(BLOCK_SWEEP))

for b in BLOCK_SWEEP:
    print(b)


Select task (A=row, B=column, C=2D): B
Selected task: B
Number of block sizes: 15
(3, 3)
(3, 5)
(3, 10)
(3, 15)
(3, 20)
(3, 30)
(3, 40)
(3, 50)
(3, 64)
(3, 80)
(3, 100)
(3, 128)
(3, 160)
(3, 200)
(32, 32)


# Clean LongBench Official Evaluation

This section is the only evaluation pipeline.

Kept unchanged:
- Llama2 model
- sparse attention pruning
- Task A/B/C block sweep

Removed:
- custom token overlap score
- duplicated evaluation functions

Added:
- official LongBench evaluator execution
- automatic score extraction
- score loss calculation


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'LongBench', 'sample_data']


In [ ]:
!find /content/LongBench -name "dataset2prompt.json"

/content/LongBench/LongBench/config/dataset2prompt.json


In [ ]:
import os
import json
import subprocess
import pandas as pd

os.makedirs("pred", exist_ok=True)
os.makedirs("results", exist_ok=True)

CONFIG_DIR = "/content/LongBench/LongBench/config"

with open(
    f"{CONFIG_DIR}/dataset2prompt.json",
    "r"
) as f:
    dataset2prompt = json.load(f)

maxlen_path = f"{CONFIG_DIR}/dataset2maxlen.json"
if os.path.exists(maxlen_path):
    with open(maxlen_path, "r") as f:
        dataset2maxlen = json.load(f)
else:
    dataset2maxlen = {}

print("LongBench evaluator ready")
print("Task max output lengths:", {k: dataset2maxlen.get(k) for k in tasks})


LongBench evaluator ready
Task max output lengths: {'narrativeqa': 128, 'qasper': 128, 'gov_report': 512}


In [ ]:
def run_longbench_eval(model_name):

    cmd = [
        "python",
        "/content/LongBench/LongBench/eval.py",
        "--model",
        model_name
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    return result.stdout

In [ ]:

def calculate_score_loss(score_dict):

    if "dense" not in score_dict:
        raise ValueError(
            "Dense baseline score is required"
        )

    dense = score_dict["dense"]

    rows = []

    for name, score in score_dict.items():

        if name == "dense":
            continue

        rows.append(
            {
                "Config": name,
                "LongBench Score": score,
                "Score Loss": dense - score
            }
        )

    return pd.DataFrame(rows)


## Final execution order

1. Select Task A/B/C.
2. Generate dense prediction:
```python
export_longbench_prediction(
    "dense",
    "pred/dense.jsonl"
)
```

3. Generate sparse predictions.
4. Run:
```python
output = run_longbench_eval()
```

5. Parse scores and calculate:
```
Score Loss = Dense Score - Sparse Score
```


In [ ]:
export_longbench_prediction(
    mode="dense"
)

Sparse attention removed
dense 0


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


dense 1


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


dense 2


...

dense 42


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


dense 43


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


dense 44
Saved: pred/dense/narrativeqa.jsonl
Saved: pred/dense/qasper.jsonl
Saved: pred/dense/gov_report.jsonl


In [ ]:
run_longbench_eval("dense")

Evaluating on: ['narrativeqa.jsonl', 'qasper.jsonl', 'gov_report.jsonl']

/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



"Evaluating on: ['narrativeqa.jsonl', 'qasper.jsonl', 'gov_report.jsonl']\n"

In [ ]:
import json

with open("pred/dense/result.json") as f:
    dense_result = json.load(f)

print(dense_result)

{'narrativeqa': 0.42, 'qasper': 1.08, 'gov_report': 34.17}


In [ ]:
# ============================================================
# COMPLETE BLOCK SWEEP
# Generate prediction + measure sparsity + evaluate score
# + save all results
# ============================================================

import os
import json
import pandas as pd


# ------------------------------------------------------------
# Check dense baseline
# ------------------------------------------------------------

dense_result_path = "pred/dense/result.json"

if not os.path.exists(dense_result_path):

    raise FileNotFoundError(
        "Dense baseline result not found:\n"
        "pred/dense/result.json\n\n"
        "Run Cell 25 first."
    )


with open(
    dense_result_path,
    "r",
    encoding="utf-8"
) as f:

    dense_scores = json.load(f)


# ------------------------------------------------------------
# Store all block results
# ------------------------------------------------------------

block_results = []

existing_results_path = "results/block_sweep_results.json"
if os.path.exists(existing_results_path):
    try:
        with open(existing_results_path, "r", encoding="utf-8") as f:
            block_results = json.load(f)
        print("Loaded existing completed blocks:", [x["Block"] for x in block_results])
    except Exception:
        block_results = []


# ------------------------------------------------------------
# Sweep
# ------------------------------------------------------------

completed_blocks = {x["Block"] for x in block_results}

for block in BLOCK_SWEEP:

    if f"{block[0]}x{block[1]}" in completed_blocks:
        print(f"SKIP completed: {block[0]}x{block[1]}")
        continue

    model_name = (
        f"sparse_{block[0]}x{block[1]}"
    )

    print("\n")
    print("=" * 70)
    print("RUNNING:", model_name)
    print("=" * 70)


    # ========================================================
    # 1. Generate sparse predictions
    # ========================================================

    export_longbench_prediction(
        mode="sparse",
        block=block
    )


    # ========================================================
    # 2. Read sparsity measured during this block
    # ========================================================

    if sparse_stats["count"] == 0:

        raise RuntimeError(
            f"No sparsity measurements collected for {model_name}"
        )


    avg_sparsity = (
        sparse_stats["sum_sparsity"]
        /
        sparse_stats["count"]
    )


    sparsity_percent = (
        avg_sparsity * 100
    )


    print(
        f"Average sparsity: "
        f"{sparsity_percent:.2f}%"
    )

    print(
        f"Measurements: "
        f"{sparse_stats['count']}"
    )


    # ========================================================
    # 3. Run LongBench evaluator
    # ========================================================

    run_longbench_eval(
        model_name
    )


    # ========================================================
    # 4. Read score generated by evaluator
    # ========================================================

    result_path = (
        f"pred/{model_name}/result.json"
    )


    if not os.path.exists(result_path):

        raise FileNotFoundError(
            f"LongBench result was not created:\n"
            f"{result_path}"
        )


    with open(
        result_path,
        "r",
        encoding="utf-8"
    ) as f:

        sparse_scores = json.load(f)


    # ========================================================
    # 5. Calculate average score across 3 tasks
    # ========================================================

    task_scores = []

    for task in dense_scores:

        if task not in sparse_scores:
            raise KeyError(
                f"{task} missing from {model_name}"
            )

        task_scores.append(
            sparse_scores[task]
        )


    sparse_avg_score = (
        sum(task_scores)
        /
        len(task_scores)
    )


    dense_task_scores = []

    for task in dense_scores:

        dense_task_scores.append(
            dense_scores[task]
        )


    dense_avg_score = (
        sum(dense_task_scores)
        /
        len(dense_task_scores)
    )


    avg_score_loss = (
        dense_avg_score
        -
        sparse_avg_score
    )


    # ========================================================
    # 6. Save result for this block
    # ========================================================

    block_result = {

        "Block":
            f"{block[0]}x{block[1]}",

        "Block Rows":
            block[0],

        "Block Cols":
            block[1],

        "Sparsity (%)":
            sparsity_percent,

        "Measurements":
            sparse_stats["count"],

        "Dense Avg Score":
            dense_avg_score,

        "Sparse Avg Score":
            sparse_avg_score,

        "Avg Score Loss":
            avg_score_loss,

        "Dense Scores":
            dense_scores,

        "Sparse Scores":
            sparse_scores
    }


    block_results.append(
        block_result
    )


    # ========================================================
    # 7. Print result immediately
    # ========================================================

    print(
        f"Sparsity      : "
        f"{sparsity_percent:.2f}%"
    )

    print(
        f"Dense Avg     : "
        f"{dense_avg_score:.4f}"
    )

    print(
        f"Sparse Avg    : "
        f"{sparse_avg_score:.4f}"
    )

    print(
        f"Average Loss  : "
        f"{avg_score_loss:.4f}"
    )


# ============================================================
# 8. Save complete JSON
# ============================================================

with open(
    "results/block_sweep_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        block_results,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 9. Create final summary table
# ============================================================

summary_rows = []


for result in block_results:

    summary_rows.append({

        "Block":
            result["Block"],

        "Sparsity (%)":
            result["Sparsity (%)"],

        "Dense Avg Score":
            result["Dense Avg Score"],

        "Sparse Avg Score":
            result["Sparse Avg Score"],

        "Avg Score Loss":
            result["Avg Score Loss"]
    })


results_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# 10. Save CSV
# ============================================================

results_df.to_csv(
    "results/block_sweep_results.csv",
    index=False
)


# ============================================================
# 11. Display final table
# ============================================================

print("\n")
print("=" * 70)
print("FINAL BLOCK SWEEP RESULTS")
print("=" * 70)

display(
    results_df
)

print("\nSaved:")
print("results/block_sweep_results.json")
print("results/block_sweep_results.csv")


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




RUNNING: sparse_3x3
Sparse attention removed
Sparse attention installed: 3x3
sparse 0


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sparse 1


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

sparse 34


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sparse 35


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sparse 36


In [ ]:
import pandas as pd

results_df = pd.read_csv(
    "results/block_sweep_results.csv"
)

display(results_df)

In [ ]:
print("Number of samples:", len(samples))

from collections import Counter
print(Counter([s["task"] for s in samples]))

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.copy()

plt.figure(figsize=(14, 6))

bars = plt.bar(
    plot_df["Block"],
    plot_df["Sparsity (%)"]
)

for bar, value in zip(
    bars,
    plot_df["Sparsity (%)"]
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.xlabel("Block Size")
plt.ylabel("Sparsity (%)")

plt.title(
    "Average Sparsity vs. Block Size"
)

plt.xticks(rotation=45)

plt.ylim(0, 100)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.copy()

plt.figure(figsize=(14, 6))

bars = plt.bar(
    plot_df["Block"],
    plot_df["Avg Score Loss"]
)

plt.axhline(
    0,
    linestyle="--",
    label="Dense baseline (loss = 0)"
)

for bar, value in zip(
    bars,
    plot_df["Avg Score Loss"]
):

    if value >= 0:
        y = value
        va = "bottom"
    else:
        y = value
        va = "top"

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        y,
        f"{value:.4f}",
        ha="center",
        va=va,
        fontsize=9
    )


plt.xlabel("Block Size")
plt.ylabel("Average Score Loss")

plt.title(
    "Average Score Loss vs. Dense Baseline"
)

plt.xticks(rotation=45)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.copy()

plt.figure(figsize=(14, 6))

bars = plt.bar(
    plot_df["Block"],
    plot_df["Sparse Avg Score"]
)

for bar, value in zip(
    bars,
    plot_df["Sparse Avg Score"]
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontsize=9
    )


dense_avg = (
    plot_df["Dense Avg Score"].iloc[0]
)


plt.axhline(
    dense_avg,
    linestyle="--",
    label=f"Dense baseline = {dense_avg:.4f}"
)


plt.xlabel("Block Size")
plt.ylabel("Average Score")
plt.title(
    "LongBench Average Score vs. Block Size"
)

plt.xticks(rotation=45)

plt.legend()
plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()